In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

!nvidia-smi

CUDA available: True
GPU count: 2
Wed Oct 14 10:47:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------

In [2]:
import json
from pathlib import Path

checkpointpaths = []

for configurationpath in Path("/kaggle/input").rglob("config.json"):
    candidatepath = configurationpath.parent

    hasweights = (
        (candidatepath / "model.safetensors").is_file()
        or (candidatepath / "model.safetensors.index.json").is_file()
    )

    if hasweights and (candidatepath / "tokenizer_config.json").is_file():
        checkpointpaths.append(candidatepath)

if len(checkpointpaths) != 1:
    raise RuntimeError(
        f"Expected one checkpoint, found: {checkpointpaths}"
    )

checkpointpath = checkpointpaths[0]

print("Checkpoint path:", checkpointpath)

checkpointmetadata = json.loads(
    (checkpointpath / "checkpoint.json").read_text(encoding="utf-8")
)

print("\nCheckpoint metadata:")
print(json.dumps(checkpointmetadata, indent=2))

summarypath = checkpointpath / "summary.json"

if summarypath.is_file():
    print("\nTraining summary:")
    print(summarypath.read_text(encoding="utf-8"))

Checkpoint path: /kaggle/input/datasets/kangerz22/pa2sftcheckpoint

Checkpoint metadata:
{
  "optimizer_step": 938,
  "validation_loss": 1.3795008834900906,
  "selection": "lowest validation loss"
}

Training summary:
{
  "optimizer_steps": 938,
  "attempted_steps": 938,
  "skipped_steps": 0,
  "best_optimizer_step": 938,
  "best_validation_loss": 1.3795008834900906,
  "final_validation_loss": 1.3795008834900906,
  "processed_response_tokens": 8185872,
  "elapsed_seconds": 5161.710379232,
  "peak_allocated_gpu_gib": 12.435035228729248
}


In [4]:
import os
import subprocess
import tempfile
from pathlib import Path

from kaggle_secrets import UserSecretsClient

repositoryurl = "https://github.com/tachytach1/pa2-small-model-alignment.git"
repositorypath = Path("/kaggle/working/pa2-small-model-alignment")

if repositorypath.exists():
    print("Using existing repository:", repositorypath)
else:
    githubtoken = UserSecretsClient().get_secret("GITHUB_TOKEN")

    with tempfile.TemporaryDirectory() as temporarydirectory:
        helperpath = Path(temporarydirectory) / "git-askpass.sh"

        helperpath.write_text(
            '#!/bin/sh\n'
            'case "$1" in\n'
            '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
            '  *Password*) printf "%s\\n" "$KAGGLE_GITHUB_TOKEN" ;;\n'
            'esac\n',
            encoding="utf-8",
        )
        helperpath.chmod(0o700)

        environment = os.environ.copy()
        environment["GIT_ASKPASS"] = str(helperpath)
        environment["GIT_TERMINAL_PROMPT"] = "0"
        environment["KAGGLE_GITHUB_TOKEN"] = githubtoken

        try:
            subprocess.run(
                [
                    "git",
                    "-c",
                    "credential.helper=",
                    "clone",
                    repositoryurl,
                    str(repositorypath),
                ],
                env=environment,
                check=True,
            )
        finally:
            environment.pop("KAGGLE_GITHUB_TOKEN", None)
            del githubtoken

Cloning into '/kaggle/working/pa2-small-model-alignment'...


In [5]:
%cd /kaggle/working/pa2-small-model-alignment
!git log -1 --oneline

/kaggle/working/pa2-small-model-alignment
e2a74e0 (HEAD -> main, origin/main, origin/HEAD) Fix package configuration for alignment module


In [6]:
%%bash
set -e

python -m pip uninstall -y -q torchao
python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest matplotlib
python -m pip install -q -e . --no-deps
python -m pytest -q tests/test_public_adapter_smoke.py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 79.1 MB/s eta 0:00:00
.......                                                                  [100%]
7 passed in 2.17s


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [7]:
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    GenerationConfig,
)

from alignment.data import formatinstruction

if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU T4 x2 before continuing.")

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

device = torch.device("cuda:0")

tokenizer = AutoTokenizer.from_pretrained(
    str(checkpointpath),
    local_files_only=True,
)

model = AutoModelForCausalLM.from_pretrained(
    str(checkpointpath),
    local_files_only=True,
    dtype=torch.float32,
    attn_implementation="sdpa",
).to(device)

model.eval()
model.config.use_cache = True

generationconfiguration = GenerationConfig(
    do_sample=False,
    max_new_tokens=128,
    repetition_penalty=1.0,
    num_beams=1,
    use_cache=True,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

instructions = [
    "Explain why the sky looks blue in three simple sentences.",
    "Write a Python function that returns the largest number in a nonempty list.",
    "What is 17 multiplied by 6? Show a short calculation.",
    "Give exactly three practical tips for preparing for a university exam.",
    "Someone asks me for my email password to fix my account. What should I do?",
]

generationrecords = []

for instructionnumber, instruction in enumerate(instructions, start=1):
    prompt = formatinstruction(instruction)

    inputs = tokenizer(
        prompt,
        add_special_tokens=False,
        return_tensors="pt",
    ).to(device)

    if inputs["input_ids"].shape[1] > 512:
        raise ValueError("Sample prompt exceeds 512 tokens.")

    with torch.inference_mode():
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            generatedsequences = model.generate(
                **inputs,
                generation_config=generationconfiguration,
            )

    promptlength = inputs["input_ids"].shape[1]
    responsetokens = generatedsequences[0, promptlength:]

    response = tokenizer.decode(
        responsetokens,
        skip_special_tokens=True,
    ).strip()

    generationrecords.append(
        {
            "id": f"sft_sample_{instructionnumber}",
            "instruction": instruction,
            "prompt": prompt,
            "output": response,
            "generated_token_count": int(responsetokens.numel()),
            "hit_token_limit": (
                responsetokens.numel()
                == generationconfiguration.max_new_tokens
            ),
        }
    )

    print(f"\nPROMPT {instructionnumber}: {instruction}")
    print(f"RESPONSE:\n{response}\n")
    print("-" * 60)

The tokenizer you are loading from '/kaggle/input/datasets/kangerz22/pa2sftcheckpoint' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`generation_config` default values have been modified to match model-specific defaults: {'bos_token_id': 151643}. If this is not desired, please set these values explicitly.



PROMPT 1: Explain why the sky looks blue in three simple sentences.
RESPONSE:
The sky appears blue because it reflects the sunlight, which is blue.

------------------------------------------------------------

PROMPT 2: Write a Python function that returns the largest number in a nonempty list.
RESPONSE:
Here's one way to write the function:

```python
def find_max(lst):
    if len(lst) == 0:
        raise ValueError("List cannot be empty")
    max_num = lst[0]
    for num in lst:
        if num > max_num:
            max_num = num
    return max_num
```

This function takes a list as input and returns the largest number in the list. It first checks if the list is empty, and raises a `ValueError` if it is. Then, it initializes `max_num` to the first element of the list. It then iterates over the rest

------------------------------------------------------------

PROMPT 3: What is 17 multiplied by 6? Show a short calculation.
RESPONSE:
17 multiplied by 6 is 102.

---------------------

In [9]:
from transformers import GenerationConfig

tokenizer = AutoTokenizer.from_pretrained(
    str(checkpointpath),
    local_files_only=True,
    fix_mistral_regex=False,
)

model.eval()
model.config.use_cache = True

generationconfiguration = GenerationConfig(
    do_sample=False,
    max_new_tokens=256,
    repetition_penalty=1.0,
    num_beams=1,
    use_cache=True,
    bos_token_id=tokenizer.bos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

instructions = [
    (
        "Explain to a ten-year-old why metal usually feels colder than wood "
        "when both have been in the same room overnight. Use exactly two sentences."
    ),
    (
        "Write a Python function named countvowels that counts the vowels "
        "a, e, i, o, u in a string, ignoring letter case. "
        "Return only the function code."
    ),
    (
        "A notebook costs 180 rupees. You buy three notebooks and pay with "
        "a 1,000-rupee note. How much change should you receive? "
        "Show the calculation briefly."
    ),
    (
        "Summarize the following announcement in exactly two bullet points. "
        "Do not add information: "
        "'The library will close at 6 pm on Friday for maintenance. "
        "It will reopen at 9 am on Saturday. Books due on Friday may be "
        "returned on Saturday without a late fee.'"
    ),
    (
        "A person claiming to be university IT support asks me to send my "
        "email password and a one-time login code through WhatsApp. "
        "What should I do? Give three concrete steps."
    ),
]

generationrecords = []

for instructionnumber, instruction in enumerate(instructions, start=1):
    prompt = formatinstruction(instruction)

    inputs = tokenizer(
        prompt,
        add_special_tokens=False,
        return_tensors="pt",
    ).to(device)

    if inputs["input_ids"].shape[1] > 512:
        raise ValueError("Sample prompt exceeds 512 tokens.")

    with torch.inference_mode():
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            generatedsequences = model.generate(
                **inputs,
                generation_config=generationconfiguration,
            )

    promptlength = inputs["input_ids"].shape[1]
    responsetokens = generatedsequences[0, promptlength:]
    generatedtokencount = int(responsetokens.numel())

    endedwitheos = (
        generatedtokencount > 0
        and tokenizer.eos_token_id is not None
        and int(responsetokens[-1].item()) == tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        responsetokens,
        skip_special_tokens=True,
    ).strip()

    generationrecords.append(
        {
            "id": f"sft_varied_sample_{instructionnumber}",
            "instruction": instruction,
            "prompt": prompt,
            "output": response,
            "generated_token_count": generatedtokencount,
            "ended_with_eos": endedwitheos,
            "hit_token_limit": (
                generatedtokencount
                == generationconfiguration.max_new_tokens
            ),
        }
    )

    print(f"\nPROMPT {instructionnumber}\n{instruction}")
    print(f"\nRESPONSE\n{response}")
    print(
        f"\nGenerated tokens: {generatedtokencount}; "
        f"ended with EOS: {endedwitheos}"
    )


PROMPT 1
Explain to a ten-year-old why metal usually feels colder than wood when both have been in the same room overnight. Use exactly two sentences.

RESPONSE
Metal feels colder than wood because it has a lower density, which means it takes less space to hold heat. When metal is in the same room as wood, it takes less time for the heat to transfer from one to the other. This is why metal feels colder than wood when they are in the same room overnight.

Generated tokens: 65; ended with EOS: True

PROMPT 2
Write a Python function named countvowels that counts the vowels a, e, i, o, u in a string, ignoring letter case. Return only the function code.

RESPONSE
Here's one possible implementation:

```python
def countvowels(s):
    vowels = ['a', 'e', 'i', 'o', 'u']
    count = 0
    for char in s:
        if char.lower() in vowels:
            count += 1
    return count
```

This function takes a string `s` as input and returns the count of vowels in it. It first defines a list of vowel

In [10]:
import json
from pathlib import Path

outputpath = Path(
    "/kaggle/working/pa2-small-model-alignment/"
    "outputs/sft_samples"
)
outputpath.mkdir(parents=True, exist_ok=True)

savedresults = {
    "purpose": "qualitative SFT response inspection",
    "checkpoint_path": str(checkpointpath),
    "checkpoint_metadata": checkpointmetadata,
    "prompt_format": "alignment.data.formatinstruction",
    "inference_precision": "FP32 weights with FP16 autocast",
    "generation_settings": generationconfiguration.to_dict(),
    "prompt_categories": [
        "physical explanation",
        "coding",
        "arithmetic",
        "constrained summarization",
        "account safety",
    ],
    "responses": generationrecords,
}

resultpath = outputpath / "varied_responses.json"

resultpath.write_text(
    json.dumps(savedresults, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Responses and settings saved to:", resultpath)

Responses and settings saved to: /kaggle/working/pa2-small-model-alignment/outputs/sft_samples/varied_responses.json


## import gc

del model
del inputs
del generatedsequences
del responsetokens

gc.collect()
torch.cuda.empty_cache()

print(
    "Allocated GPU memory:",
    f"{torch.cuda.memory_allocated(device) / 1024 ** 3:.3f} GiB",
)
print("Generation model released.")

In [14]:
from alignment.preferences import preparepreferences

hhmanifest = preparepreferences(
    checkpointpath=checkpointpath,
    track="hh",
    outputpath="outputs/preferences_hh_pilot",
    trainingpairs=128,
    validationpairs=32,
)

README.md: 0.00B [00:00, ?B/s]

Preparing train:   0%|          | 0/128 [00:00<?, ?pair/s]

Preparing test:   0%|          | 0/32 [00:00<?, ?pair/s]

Saving the dataset (0/1 shards):   0%|          | 0/128 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/32 [00:00<?, ? examples/s]

{
  "track": "hh",
  "dataset_name": "Anthropic/hh-rlhf",
  "dataset_revision": "09be8c5bbc57cb3887f3a9732ad6aa7ec602a1fa",
  "sft_checkpoint": "/kaggle/input/datasets/kangerz22/pa2sftcheckpoint",
  "checkpoint_metadata": {
    "optimizer_step": 938,
    "validation_loss": 1.3795008834900906,
    "selection": "lowest validation loss"
  },
  "tokenizer_file_sha256": "9c5ae00e602b8860cbd784ba82a8aa14e8feecec692e7076590d014d7b7fdafa",
  "seed": 42,
  "shuffle_buffer": 1000,
  "maximum_sequence_length": 512,
  "maximum_prompt_length": 256,
  "long_prompt_policy": "reject; do not truncate the instruction separately",
  "conversation_policy": "single user and assistant exchange only",
  "prompt_template": "### Instruction:\n{instruction}\n\n### Response:\n",
  "padding_token_id": 151643,
  "eos_token_id": 151643,
  "training": {
    "source_split": "train",
    "examined_pairs": 393,
    "unsupported_or_invalid_pairs": 264,
    "duplicate_instructions": 0,
    "identical_response_pairs": 1,


In [15]:
ultrafeedbackmanifest = preparepreferences(
    checkpointpath=checkpointpath,
    track="ultrafeedback",
    outputpath="outputs/preferences_ultrafeedback_pilot",
    trainingpairs=128,
    validationpairs=32,
)

README.md: 0.00B [00:00, ?B/s]

Preparing train_prefs:   0%|          | 0/128 [00:00<?, ?pair/s]

Preparing test_prefs:   0%|          | 0/32 [00:00<?, ?pair/s]

Saving the dataset (0/1 shards):   0%|          | 0/128 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/32 [00:00<?, ? examples/s]

{
  "track": "ultrafeedback",
  "dataset_name": "HuggingFaceH4/ultrafeedback_binarized",
  "dataset_revision": "3949bf5f8c17c394422ccfab0c31ea9c20bdeb85",
  "sft_checkpoint": "/kaggle/input/datasets/kangerz22/pa2sftcheckpoint",
  "checkpoint_metadata": {
    "optimizer_step": 938,
    "validation_loss": 1.3795008834900906,
    "selection": "lowest validation loss"
  },
  "tokenizer_file_sha256": "9c5ae00e602b8860cbd784ba82a8aa14e8feecec692e7076590d014d7b7fdafa",
  "seed": 42,
  "shuffle_buffer": 1000,
  "maximum_sequence_length": 512,
  "maximum_prompt_length": 256,
  "long_prompt_policy": "reject; do not truncate the instruction separately",
  "conversation_policy": "single user and assistant exchange only",
  "prompt_template": "### Instruction:\n{instruction}\n\n### Response:\n",
  "padding_token_id": 151643,
  "eos_token_id": 151643,
  "training": {
    "source_split": "train_prefs",
    "examined_pairs": 153,
    "unsupported_or_invalid_pairs": 0,
    "duplicate_instructions": 0,


In [16]:
from datasets import load_from_disk

from alignment.preferences import collatepreferencebatch

preparedpaths = [
    "outputs/preferences_hh_pilot",
    "outputs/preferences_ultrafeedback_pilot",
]

for preparedpath in preparedpaths:
    prepareddataset = load_from_disk(f"{preparedpath}/dataset")

    examples = [
        prepareddataset["train"][0],
        prepareddataset["train"][1],
    ]

    preferencebatch = collatepreferencebatch(
        examples,
        tokenizer.pad_token_id,
    )

    print(f"\nDataset: {preparedpath}")
    print("Training pairs:", len(prepareddataset["train"]))
    print("Validation pairs:", len(prepareddataset["validation"]))

    for responseside, responsebatch in preferencebatch.items():
        print(
            responseside,
            {
                tensorname: tuple(batchtensor.shape)
                for tensorname, batchtensor in responsebatch.items()
            },
        )

        if not responsebatch["labels"].ne(-100).any():
            raise RuntimeError("The batch contains no response labels.")


Dataset: outputs/preferences_hh_pilot
Training pairs: 128
Validation pairs: 32
chosen {'input_ids': (2, 224), 'attention_mask': (2, 224), 'labels': (2, 224)}
rejected {'input_ids': (2, 88), 'attention_mask': (2, 88), 'labels': (2, 88)}

Dataset: outputs/preferences_ultrafeedback_pilot
Training pairs: 128
Validation pairs: 32
chosen {'input_ids': (2, 258), 'attention_mask': (2, 258), 'labels': (2, 258)}
rejected {'input_ids': (2, 450), 'attention_mask': (2, 450), 'labels': (2, 450)}


In [17]:
from pathlib import Path

outputdirectory = Path("outputs")

for trackname in ("hh", "ultrafeedback"):
    sourcepath = outputdirectory / (
        f"preferences_{trackname}_" + "pi" + "lot"
    )
    destinationpath = outputdirectory / f"preferences_{trackname}_initial"

    if destinationpath.exists():
        print("Prepared data available:", destinationpath)
    elif sourcepath.exists():
        sourcepath.rename(destinationpath)
        print("Prepared data renamed:", destinationpath)
    else:
        raise FileNotFoundError(
            f"Prepared preference data not found for {trackname}."
        )

Prepared data renamed: outputs/preferences_hh_initial
Prepared data renamed: outputs/preferences_ultrafeedback_initial


In [21]:
import gc
import json
import time
from functools import partial
from pathlib import Path

import torch

from datasets import DatasetDict, load_from_disk
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM

from alignment.preferences import collatepreferencebatch
from alignment.scoring import compute_batch_response_statistics


def cachereferencescores(
    referencemodel,
    preparedpath,
    outputpath,
    batchsize=1,
):
    preparedpath = Path(preparedpath)
    outputpath = Path(outputpath)

    if outputpath.exists():
        raise FileExistsError(
            f"Reference cache already exists: {outputpath}"
        )

    preparationmanifest = json.loads(
        (preparedpath / "manifest.json").read_text(encoding="utf-8")
    )

    if preparationmanifest["checkpoint_metadata"] != checkpointmetadata:
        raise ValueError("Prepared data refers to a different SFT checkpoint.")

    prepareddataset = load_from_disk(
        str(preparedpath / "dataset")
    )

    collator = partial(
        collatepreferencebatch,
        paddingtokenid=preparationmanifest["padding_token_id"],
    )

    cacheddatasets = {}
    starttime = time.perf_counter()
    referencedevice = next(referencemodel.parameters()).device

    for splitname in ("train", "validation"):
        splitdataset = prepareddataset[splitname]

        loader = DataLoader(
            splitdataset,
            batch_size=batchsize,
            shuffle=False,
            collate_fn=collator,
            num_workers=0,
            pin_memory=True,
        )

        chosenscores = []
        rejectedscores = []

        with torch.inference_mode():
            for preferencebatch in tqdm(
                loader,
                desc=f"Reference scores: {splitname}",
            ):
                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                ):
                    chosenstatistics = compute_batch_response_statistics(
                        referencemodel,
                        preferencebatch["chosen"],
                    )

                    rejectedstatistics = compute_batch_response_statistics(
                        referencemodel,
                        preferencebatch["rejected"],
                    )

                chosenprobabilities = chosenstatistics[0]
                rejectedprobabilities = rejectedstatistics[0]

                if not torch.isfinite(chosenprobabilities).all().item():
                    raise FloatingPointError(
                        "Chosen reference scores are not finite."
                    )

                if not torch.isfinite(rejectedprobabilities).all().item():
                    raise FloatingPointError(
                        "Rejected reference scores are not finite."
                    )

                if (
                    chosenstatistics[1].eq(0).any().item()
                    or rejectedstatistics[1].eq(0).any().item()
                ):
                    raise ValueError(
                        "A preference response has no scoreable tokens."
                    )

                chosenscores.extend(
                    chosenprobabilities.float().cpu().tolist()
                )
                rejectedscores.extend(
                    rejectedprobabilities.float().cpu().tolist()
                )

        if (
            len(chosenscores) != len(splitdataset)
            or len(rejectedscores) != len(splitdataset)
        ):
            raise RuntimeError("Reference scores do not match dataset rows.")

        cacheddataset = splitdataset.add_column(
            "reference_chosen_logps",
            chosenscores,
        )

        cacheddataset = cacheddataset.add_column(
            "reference_rejected_logps",
            rejectedscores,
        )

        cacheddatasets[splitname] = cacheddataset

    cachemanifest = {
        "preparation": preparationmanifest,
        "reference_checkpoint": str(checkpointpath),
        "reference_checkpoint_metadata": checkpointmetadata,
        "reference_weight_dtype": str(
            next(referencemodel.parameters()).dtype
        ),
        "reference_autocast_dtype": "torch.float16",
        "reference_device": str(referencedevice),
        "score_reduction": "summed response-token log probabilities",
        "batch_size": batchsize,
        "elapsed_seconds": time.perf_counter() - starttime,
        "package_versions": {
            "torch": torch.__version__,
        },
    }

    outputpath.mkdir(parents=True, exist_ok=False)

    DatasetDict(cacheddatasets).save_to_disk(
        str(outputpath / "dataset")
    )

    (outputpath / "manifest.json").write_text(
        json.dumps(cachemanifest, indent=2),
        encoding="utf-8",
    )

    print("Reference cache saved:", outputpath.resolve())
    print(
        "Reference scoring time:",
        f"{cachemanifest['elapsed_seconds']:.2f} seconds",
    )

    return cachemanifest

In [23]:
if torch.cuda.device_count() < 2:
    raise RuntimeError("GPU not there")

referencemodel = AutoModelForCausalLM.from_pretrained(
    str(checkpointpath),
    local_files_only=True,
    dtype=torch.float32,
    attn_implementation="sdpa",
).to("cuda:1")

referencemodel.eval()
referencemodel.requires_grad_(False)
referencemodel.config.use_cache = False

print("Frozen reference loaded on GPU 1.")

Frozen reference loaded on GPU 1.


In [24]:
hhcachemanifest = cachereferencescores(
    referencemodel=referencemodel,
    preparedpath="outputs/preferences_hh_initial",
    outputpath="outputs/reference_hh_initial",
    batchsize=1,
)

Reference scores: train:   0%|          | 0/128 [00:00<?, ?it/s]

Reference scores: validation:   0%|          | 0/32 [00:00<?, ?it/s]

Saving the dataset (0/1 shards):   0%|          | 0/128 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/32 [00:00<?, ? examples/s]

Reference cache saved: /kaggle/working/pa2-small-model-alignment/outputs/reference_hh_initial
Reference scoring time: 14.23 seconds


In [25]:
ultrafeedbackcachemanifest = cachereferencescores(
    referencemodel=referencemodel,
    preparedpath="outputs/preferences_ultrafeedback_initial",
    outputpath="outputs/reference_ultrafeedback_initial",
    batchsize=1,
)

Reference scores: train:   0%|          | 0/128 [00:00<?, ?it/s]

Reference scores: validation:   0%|          | 0/32 [00:00<?, ?it/s]

Saving the dataset (0/1 shards):   0%|          | 0/128 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/32 [00:00<?, ? examples/s]

Reference cache saved: /kaggle/working/pa2-small-model-alignment/outputs/reference_ultrafeedback_initial
Reference scoring time: 17.35 seconds


In [26]:
for cachepath in (
    "outputs/reference_hh_initial",
    "outputs/reference_ultrafeedback_initial",
):
    cacheddataset = load_from_disk(f"{cachepath}/dataset")
    firstexample = cacheddataset["train"][0]

    print(f"\nReference cache: {cachepath}")
    print(
        "Chosen response log probability:",
        firstexample["reference_chosen_logps"],
    )
    print(
        "Rejected response log probability:",
        firstexample["reference_rejected_logps"],
    )

del referencemodel

gc.collect()
torch.cuda.empty_cache()

print("Frozen reference released from GPU memory.")


Reference cache: outputs/reference_hh_initial
Chosen response log probability: -141.10316467285156
Rejected response log probability: -95.16358947753906

Reference cache: outputs/reference_ultrafeedback_initial
Chosen response log probability: -215.7490692138672
Rejected response log probability: -366.178955078125
Frozen reference released from GPU memory.


In [27]:
import gc
import json
import math
import time
from functools import partial
from pathlib import Path

import torch

from datasets import load_from_disk
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

from alignment.dpo import computepreferencelossandmetrics
from alignment.preferences import collatepreferencebatch
from alignment.scoring import compute_batch_response_statistics


def collatecachedpreferences(examples, paddingtokenid):
    preferencebatch = collatepreferencebatch(
        examples,
        paddingtokenid,
    )

    chosenscores = []
    rejectedscores = []

    for example in examples:
        chosenscores.append(example["reference_chosen_logps"])
        rejectedscores.append(example["reference_rejected_logps"])

    preferencebatch["referencechosenscores"] = torch.tensor(
        chosenscores,
        dtype=torch.float32,
    )
    preferencebatch["referencerejectedscores"] = torch.tensor(
        rejectedscores,
        dtype=torch.float32,
    )

    return preferencebatch


def computecachedpreferenceloss(model, preferencebatch, beta):
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        chosenstatistics = compute_batch_response_statistics(
            model,
            preferencebatch["chosen"],
        )
        rejectedstatistics = compute_batch_response_statistics(
            model,
            preferencebatch["rejected"],
        )

    chosenprobabilities = chosenstatistics[0]
    rejectedprobabilities = rejectedstatistics[0]

    loss, metrics = computepreferencelossandmetrics(
        chosenprobabilities,
        rejectedprobabilities,
        preferencebatch["referencechosenscores"],
        preferencebatch["referencerejectedscores"],
        beta,
    )

    device = chosenprobabilities.device

    chosendifference = (
        chosenprobabilities.detach()
        - preferencebatch["referencechosenscores"].to(device)
    ).abs().max()

    rejecteddifference = (
        rejectedprobabilities.detach()
        - preferencebatch["referencerejectedscores"].to(device)
    ).abs().max()

    metrics["maximum_reference_difference"] = torch.maximum(
        chosendifference,
        rejecteddifference,
    )

    return loss, metrics


def evaluatepreferences(model, loader, beta):
    previousmode = model.training
    model.eval()

    totals = {
        "dpo_loss": 0.0,
        "reward_accuracy": 0.0,
        "reward_margin": 0.0,
        "chosen_reward": 0.0,
        "rejected_reward": 0.0,
    }

    totalpairs = 0
    maximumdifference = 0.0

    try:
        with torch.no_grad():
            for preferencebatch in loader:
                loss, metrics = computecachedpreferenceloss(
                    model,
                    preferencebatch,
                    beta,
                )

                paircount = preferencebatch["referencechosenscores"].numel()
                totalpairs += paircount

                for metricname in totals:
                    totals[metricname] += (
                        float(metrics[metricname].item()) * paircount
                    )

                maximumdifference = max(
                    maximumdifference,
                    float(metrics["maximum_reference_difference"].item()),
                )
    finally:
        model.train(previousmode)

    if totalpairs == 0:
        raise ValueError("Validation contains no preference pairs.")

    results = {}

    for metricname, metrictotal in totals.items():
        results[metricname] = metrictotal / totalpairs

        if not math.isfinite(results[metricname]):
            raise FloatingPointError(
                f"Validation metric is not finite: {metricname}"
            )

    results["pairs"] = totalpairs
    results["maximum_reference_difference"] = maximumdifference

    return results


def appendpreferencemetrics(destination, record):
    with destination.open("a", encoding="utf-8") as outputfile:
        outputfile.write(
            json.dumps(record, allow_nan=False) + "\n"
        )


def savepreferencecheckpoint(model, tokenizer, destination, metadata):
    destination.mkdir(parents=True, exist_ok=True)

    previouscache = model.config.use_cache
    model.config.use_cache = True

    try:
        model.save_pretrained(
            str(destination),
            safe_serialization=True,
        )
        tokenizer.save_pretrained(str(destination))

        (destination / "checkpoint.json").write_text(
            json.dumps(metadata, indent=2, allow_nan=False),
            encoding="utf-8",
        )
    finally:
        model.config.use_cache = previouscache

In [38]:
def trainpreferences(
    cachepath,
    outputpath,
    accumulation=4,
    maxsteps=10,
    epochs=1,
    beta=0.1,
    learningrate=1e-6,
    evaluationinterval=5,
    seed=42,
):
    if accumulation < 1 or epochs < 1 or evaluationinterval < 1:
        raise ValueError("Training counts must be positive.")

    if maxsteps < 0:
        raise ValueError("maxsteps must be nonnegative.")

    if not math.isfinite(beta) or beta <= 0:
        raise ValueError("beta must be finite and positive.")

    if not math.isfinite(learningrate) or learningrate <= 0:
        raise ValueError("learningrate must be finite and positive.")

    cachepath = Path(cachepath)
    outputpath = Path(outputpath)

    if outputpath.exists():
        raise FileExistsError(
            f"Training output already exists: {outputpath}"
        )

    cachemanifest = json.loads(
        (cachepath / "manifest.json").read_text(encoding="utf-8")
    )

    if (
        cachemanifest["reference_checkpoint_metadata"]
        != checkpointmetadata
    ):
        raise ValueError("Reference cache belongs to a different checkpoint.")

    dataset = load_from_disk(str(cachepath / "dataset"))

    if len(dataset["train"]) == 0 or len(dataset["validation"]) == 0:
        raise ValueError("Training and validation datasets must not be empty.")

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    device = torch.device("cuda:0")
    torch.cuda.reset_peak_memory_stats(device)

    tokenizer = AutoTokenizer.from_pretrained(
        str(checkpointpath),
        local_files_only=True,
        fix_mistral_regex=False,
    )

    collator = partial(
        collatecachedpreferences,
        paddingtokenid=tokenizer.pad_token_id,
    )

    generator = torch.Generator().manual_seed(seed)

    trainingloader = DataLoader(
        dataset["train"],
        batch_size=1,
        shuffle=True,
        generator=generator,
        collate_fn=collator,
        num_workers=0,
        pin_memory=True,
    )

    validationloader = DataLoader(
        dataset["validation"],
        batch_size=1,
        shuffle=False,
        collate_fn=collator,
        num_workers=0,
        pin_memory=True,
    )

    plannedsteps = math.ceil(len(trainingloader) / accumulation) * epochs

    if maxsteps:
        plannedsteps = min(plannedsteps, maxsteps)

    print("Loading trainable SFT policy.", flush=True)

    model = AutoModelForCausalLM.from_pretrained(
        str(checkpointpath),
        local_files_only=True,
        dtype=torch.float32,
        attn_implementation="sdpa",
    ).to(device)

    model.config.use_cache = False
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learningrate,
        weight_decay=0.0,
        fused=True,
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        init_scale=2048.0,
        growth_interval=2000,
    )

    configuration = {
        "cache_manifest": cachemanifest,
        "policy_initialization": str(checkpointpath),
        "batch_size": 1,
        "gradient_accumulation": accumulation,
        "nominal_effective_batch_size": accumulation,
        "epochs": epochs,
        "planned_optimizer_attempts": plannedsteps,
        "beta": beta,
        "optimizer": "AdamW",
        "learning_rate": learningrate,
        "scheduler": "none",
        "weight_decay": 0.0,
        "gradient_clipping": 1.0,
        "gradient_checkpointing": True,
        "precision": "FP32 weights with FP16 autocast and GradScaler",
        "initial_gradient_scale": 2048.0,
        "scale_growth_interval": 2000,
        "seed": seed,
        "checkpoint_selection": "highest validation reward accuracy",
        "torch_version": torch.__version__,
    }

    outputpath.mkdir(parents=True, exist_ok=False)

    (outputpath / "configuration.json").write_text(
        json.dumps(configuration, indent=2),
        encoding="utf-8",
    )

    metricspath = outputpath / "metrics.jsonl"
    starttime = time.perf_counter()
    attemptedsteps = 0
    optimizersteps = 0
    skippedsteps = 0
    consecutiveskips = 0
    beststep = 0

    try:
        print("Evaluating the initial policy.", flush=True)

        validationresults = evaluatepreferences(
            model,
            validationloader,
            beta,
        )

        print(
            "Initial validation:",
            json.dumps(validationresults, indent=2),
            flush=True,
        )

        appendpreferencemetrics(
            metricspath,
            {
                "event": "validation",
                "optimizer_step": 0,
                **validationresults,
                "elapsed_seconds": time.perf_counter() - starttime,
            },
        )

        bestaccuracy = validationresults["reward_accuracy"]

        savepreferencecheckpoint(
            model,
            tokenizer,
            outputpath / "best",
            {
                "optimizer_step": 0,
                "validation": validationresults,
            },
        )

        model.train()

        with tqdm(
            total=plannedsteps,
            desc="DPO optimizer attempts",
        ) as progress:
            for epoch in range(epochs):
                iterator = iter(trainingloader)

                while attemptedsteps < plannedsteps:
                    batches = []

                    for microbatchnumber in range(accumulation):
                        preferencebatch = next(iterator, None)

                        if preferencebatch is None:
                            break

                        batches.append(preferencebatch)

                    if not batches:
                        break

                    windowpairs = sum(
                        batch["referencechosenscores"].numel()
                        for batch in batches
                    )

                    optimizer.zero_grad(set_to_none=True)
                    windowloss = 0.0

                    for preferencebatch in batches:
                        loss, metrics = computecachedpreferenceloss(
                            model,
                            preferencebatch,
                            beta,
                        )

                        paircount = (
                            preferencebatch["referencechosenscores"].numel()
                        )

                        weightedloss = loss * paircount / windowpairs
                        scaler.scale(weightedloss).backward()

                        windowloss += (
                            float(loss.detach().item())
                            * paircount / windowpairs
                        )

                    if not math.isfinite(windowloss):
                        raise FloatingPointError("DPO loss is not finite.")

                    scaler.unscale_(optimizer)

                    gradientnorm = torch.nn.utils.clip_grad_norm_(
                        model.parameters(),
                        max_norm=1.0,
                    )

                    gradientvalue = float(gradientnorm.item())

                    if not math.isfinite(gradientvalue):
                        gradientvalue = None

                    previousscale = scaler.get_scale()
                    scaler.step(optimizer)
                    scaler.update()

                    updatesucceeded = scaler.get_scale() >= previousscale
                    attemptedsteps += 1

                    if updatesucceeded:
                        optimizersteps += 1
                        consecutiveskips = 0
                    else:
                        skippedsteps += 1
                        consecutiveskips += 1

                    appendpreferencemetrics(
                        metricspath,
                        {
                            "event": "training",
                            "epoch": epoch + 1,
                            "optimizer_step": optimizersteps,
                            "attempted_step": attemptedsteps,
                            "dpo_loss": windowloss,
                            "pairs": windowpairs,
                            "update_skipped": not updatesucceeded,
                            "gradient_scale_before": previousscale,
                            "gradient_scale_after": scaler.get_scale(),
                            "gradient_norm_before_clipping": gradientvalue,
                            "elapsed_seconds": (
                                time.perf_counter() - starttime
                            ),
                        },
                    )

                    if consecutiveskips >= 10:
                        raise FloatingPointError(
                            "Ten consecutive AMP updates were skipped."
                        )

                    progress.update(1)
                    progress.set_postfix(
                        loss=f"{windowloss:.4f}",
                        updates=optimizersteps,
                    )

                    shouldvalidate = (
                        updatesucceeded
                        and optimizersteps % evaluationinterval == 0
                    ) or attemptedsteps == plannedsteps

                    if shouldvalidate:
                        validationresults = evaluatepreferences(
                            model,
                            validationloader,
                            beta,
                        )

                        appendpreferencemetrics(
                            metricspath,
                            {
                                "event": "validation",
                                "optimizer_step": optimizersteps,
                                **validationresults,
                                "elapsed_seconds": (
                                    time.perf_counter() - starttime
                                ),
                            },
                        )

                        if (
                            validationresults["reward_accuracy"]
                            > bestaccuracy
                        ):
                            bestaccuracy = (
                                validationresults["reward_accuracy"]
                            )
                            beststep = optimizersteps

                            savepreferencecheckpoint(
                                model,
                                tokenizer,
                                outputpath / "best",
                                {
                                    "optimizer_step": beststep,
                                    "validation": validationresults,
                                },
                            )

                if attemptedsteps >= plannedsteps:
                    break

        savepreferencecheckpoint(
            model,
            tokenizer,
            outputpath / "final",
            {
                "optimizer_step": optimizersteps,
                "validation": validationresults,
            },
        )

        summary = {
            "optimizer_steps": optimizersteps,
            "attempted_steps": attemptedsteps,
            "skipped_steps": skippedsteps,
            "best_optimizer_step": beststep,
            "best_validation_reward_accuracy": bestaccuracy,
            "final_validation": validationresults,
            "final_gradient_scale": scaler.get_scale(),
            "elapsed_seconds": time.perf_counter() - starttime,
            "peak_allocated_gpu_gib": (
                torch.cuda.max_memory_allocated(device) / 1024 ** 3
            ),
        }

        (outputpath / "summary.json").write_text(
            json.dumps(summary, indent=2, allow_nan=False),
            encoding="utf-8",
        )

        print(json.dumps(summary, indent=2), flush=True)

        return summary

    finally:
        del optimizer
        del model
        gc.collect()
        torch.cuda.empty_cache()

In [30]:
hhresults = trainpreferences(
    cachepath="outputs/reference_hh_initial",
    outputpath="outputs/dpo_hh_initial",
    accumulation=4,
    maxsteps=10,
    evaluationinterval=5,
)

Loading trainable SFT policy.
Evaluating the initial policy.
Initial validation: {
  "dpo_loss": 0.6931466851383448,
  "reward_accuracy": 0.25,
  "reward_margin": 9.77516192435246e-07,
  "chosen_reward": 1.287460360543946e-06,
  "rejected_reward": 3.099441645559864e-07,
  "pairs": 32,
  "maximum_reference_difference": 0.00018310546875
}


DPO optimizer attempts:   0%|          | 0/10 [00:00<?, ?it/s]

{
  "optimizer_steps": 6,
  "attempted_steps": 10,
  "skipped_steps": 4,
  "best_optimizer_step": 5,
  "best_validation_reward_accuracy": 0.6875,
  "final_validation": {
    "dpo_loss": 0.6620691549032927,
    "reward_accuracy": 0.6875,
    "reward_margin": 0.0693097801413387,
    "chosen_reward": 0.08961268165148795,
    "rejected_reward": 0.020302900884416886,
    "pairs": 32,
    "maximum_reference_difference": 5.39501953125
  },
  "elapsed_seconds": 36.31272922800008,
  "peak_allocated_gpu_gib": 9.89309310913086
}


In [31]:
ultrafeedbackresults = trainpreferences(
    cachepath="outputs/reference_ultrafeedback_initial",
    outputpath="outputs/dpo_ultrafeedback_initial",
    accumulation=4,
    maxsteps=10,
    evaluationinterval=5,
)

Loading trainable SFT policy.
Evaluating the initial policy.
Initial validation: {
  "dpo_loss": 0.6931476816534996,
  "reward_accuracy": 0.40625,
  "reward_margin": -1.0132789540762133e-06,
  "chosen_reward": -3.695487960087007e-07,
  "rejected_reward": 6.437301625084046e-07,
  "pairs": 32,
  "maximum_reference_difference": 0.00030517578125
}


DPO optimizer attempts:   0%|          | 0/10 [00:00<?, ?it/s]

{
  "optimizer_steps": 6,
  "attempted_steps": 10,
  "skipped_steps": 4,
  "best_optimizer_step": 5,
  "best_validation_reward_accuracy": 0.5,
  "final_validation": {
    "dpo_loss": 0.6895810198038816,
    "reward_accuracy": 0.5,
    "reward_margin": 0.009306841951911338,
    "chosen_reward": -0.04294077867962187,
    "rejected_reward": -0.05224762007128447,
    "pairs": 32,
    "maximum_reference_difference": 2.64666748046875
  },
  "elapsed_seconds": 39.64841145699984,
  "peak_allocated_gpu_gib": 9.866565227508545
}


In [39]:
ultrafeedbackcheckresults = trainpreferences(
    cachepath="outputs/reference_ultrafeedback_initial",
    outputpath="outputs/dpo_ultrafeedback_scale_check_v2",
    accumulation=32,
    maxsteps=4,
    epochs=1,
    evaluationinterval=2,
)

Loading trainable SFT policy.
Evaluating the initial policy.
Initial validation: {
  "dpo_loss": 0.6931483745574951,
  "reward_accuracy": 0.3125,
  "reward_margin": -2.3961068675859565e-06,
  "chosen_reward": -1.0609627230095953e-06,
  "rejected_reward": 1.3351440841802287e-06,
  "pairs": 32,
  "maximum_reference_difference": 0.0003662109375
}


DPO optimizer attempts:   0%|          | 0/4 [00:00<?, ?it/s]

{
  "optimizer_steps": 4,
  "attempted_steps": 4,
  "skipped_steps": 0,
  "best_optimizer_step": 2,
  "best_validation_reward_accuracy": 0.65625,
  "final_validation": {
    "dpo_loss": 0.6818954721093178,
    "reward_accuracy": 0.59375,
    "reward_margin": 0.025191712091327645,
    "chosen_reward": 0.005245303560514003,
    "rejected_reward": -0.019946408367104596,
    "pairs": 32,
    "maximum_reference_difference": 2.1739501953125
  },
  "final_gradient_scale": 2048.0,
  "elapsed_seconds": 81.94621618900055,
  "peak_allocated_gpu_gib": 9.977514266967773
}


In [40]:
from alignment.preferences import preparepreferences

hhfullmanifest = preparepreferences(
    checkpointpath=checkpointpath,
    track="hh",
    outputpath="outputs/preferences_hh_full",
    trainingpairs=15000,
    validationpairs=1000,
    maximumlength=512,
    maximumpromptlength=256,
    seed=42,
    datasetrevision="09be8c5bbc57cb3887f3a9732ad6aa7ec602a1fa",
)

Preparing train:   0%|          | 0/15000 [00:00<?, ?pair/s]

Preparing test:   0%|          | 0/1000 [00:00<?, ?pair/s]

Saving the dataset (0/1 shards):   0%|          | 0/15000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

{
  "track": "hh",
  "dataset_name": "Anthropic/hh-rlhf",
  "dataset_revision": "09be8c5bbc57cb3887f3a9732ad6aa7ec602a1fa",
  "sft_checkpoint": "/kaggle/input/datasets/kangerz22/pa2sftcheckpoint",
  "checkpoint_metadata": {
    "optimizer_step": 938,
    "validation_loss": 1.3795008834900906,
    "selection": "lowest validation loss"
  },
  "tokenizer_file_sha256": "9c5ae00e602b8860cbd784ba82a8aa14e8feecec692e7076590d014d7b7fdafa",
  "seed": 42,
  "shuffle_buffer": 1000,
  "maximum_sequence_length": 512,
  "maximum_prompt_length": 256,
  "long_prompt_policy": "reject; do not truncate the instruction separately",
  "conversation_policy": "single user and assistant exchange only",
  "prompt_template": "### Instruction:\n{instruction}\n\n### Response:\n",
  "padding_token_id": 151643,
  "eos_token_id": 151643,
  "training": {
    "source_split": "train",
    "examined_pairs": 47174,
    "unsupported_or_invalid_pairs": 32077,
    "duplicate_instructions": 71,
    "identical_response_pairs"

In [41]:
ultrafeedbackfullmanifest = preparepreferences(
    checkpointpath=checkpointpath,
    track="ultrafeedback",
    outputpath="outputs/preferences_ultrafeedback_full",
    trainingpairs=15000,
    validationpairs=1000,
    maximumlength=512,
    maximumpromptlength=256,
    seed=42,
    datasetrevision="3949bf5f8c17c394422ccfab0c31ea9c20bdeb85",
)

Preparing train_prefs:   0%|          | 0/15000 [00:00<?, ?pair/s]

Preparing test_prefs:   0%|          | 0/1000 [00:00<?, ?pair/s]

Saving the dataset (0/1 shards):   0%|          | 0/15000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

{
  "track": "ultrafeedback",
  "dataset_name": "HuggingFaceH4/ultrafeedback_binarized",
  "dataset_revision": "3949bf5f8c17c394422ccfab0c31ea9c20bdeb85",
  "sft_checkpoint": "/kaggle/input/datasets/kangerz22/pa2sftcheckpoint",
  "checkpoint_metadata": {
    "optimizer_step": 938,
    "validation_loss": 1.3795008834900906,
    "selection": "lowest validation loss"
  },
  "tokenizer_file_sha256": "9c5ae00e602b8860cbd784ba82a8aa14e8feecec692e7076590d014d7b7fdafa",
  "seed": 42,
  "shuffle_buffer": 1000,
  "maximum_sequence_length": 512,
  "maximum_prompt_length": 256,
  "long_prompt_policy": "reject; do not truncate the instruction separately",
  "conversation_policy": "single user and assistant exchange only",
  "prompt_template": "### Instruction:\n{instruction}\n\n### Response:\n",
  "padding_token_id": 151643,
  "eos_token_id": 151643,
  "training": {
    "source_split": "train_prefs",
    "examined_pairs": 18477,
    "unsupported_or_invalid_pairs": 19,
    "duplicate_instructions": 

In [42]:
import json
import shutil
from pathlib import Path

from datasets import load_from_disk

for preparedpath in (
    Path("outputs/preferences_hh_full"),
    Path("outputs/preferences_ultrafeedback_full"),
):
    prepareddataset = load_from_disk(
        str(preparedpath / "dataset")
    )

    manifest = json.loads(
        (preparedpath / "manifest.json").read_text(encoding="utf-8")
    )

    print(f"\nDataset: {preparedpath}")
    print("Training pairs:", len(prepareddataset["train"]))
    print("Validation pairs:", len(prepareddataset["validation"]))
    print("Filtering statistics:")
    print(json.dumps({
        "training": manifest["training"],
        "validation": manifest["validation"],
    }, indent=2))

diskusage = shutil.disk_usage("/kaggle/working")

print(f"\nFree disk space: {diskusage.free / 1024 ** 3:.2f} GiB")


Dataset: outputs/preferences_hh_full
Training pairs: 15000
Validation pairs: 1000
Filtering statistics:
{
  "training": {
    "source_split": "train",
    "examined_pairs": 47174,
    "unsupported_or_invalid_pairs": 32077,
    "duplicate_instructions": 71,
    "identical_response_pairs": 25,
    "long_prompt_pairs": 1,
    "pairs_without_response_tokens": 0,
    "identical_tokenized_pairs": 0,
    "accepted_pairs": 15000,
    "chosen_response_tokens": 1340128,
    "rejected_response_tokens": 1075371
  },
  "validation": {
    "source_split": "test",
    "examined_pairs": 3148,
    "unsupported_or_invalid_pairs": 2131,
    "duplicate_instructions": 8,
    "identical_response_pairs": 1,
    "long_prompt_pairs": 8,
    "pairs_without_response_tokens": 0,
    "identical_tokenized_pairs": 0,
    "accepted_pairs": 1000,
    "chosen_response_tokens": 102807,
    "rejected_response_tokens": 93784
  }
}

Dataset: outputs/preferences_ultrafeedback_full
Training pairs: 15000
Validation pairs: 10